# 05 — Seasonal ARIMA (SARIMA) Forecasting
## Corporación Favorita Grocery Sales Forecasting

### Purpose of this Notebook
This notebook builds and evaluates a **Seasonal Autoregressive Integrated Moving Average (SARIMA)** model:
1. **Capturing Weekly Seasonality:** Modeling 7-day cyclical shopping behavior using seasonal autoregressive and moving average terms: $(P, D, Q)_s$ where $s=7$.
2. **National & Granular Benchmarks:** Fitting $\text{SARIMA}(2,1,2)(1,1,1)_7$ on national sales and Store 1 / Grocery I.
3. **Walk-Forward Cross-Validation:** Performing 5-fold expanding window cross-validation (`TimeSeriesSplit`) with a 30-day forecast horizon.
4. **Zero-Safe Error Metrics:** Evaluating forecasts using MAE, RMSE, Weighted MAPE (WMAPE), and Symmetric MAPE (SMAPE).
5. **Artifact Export:** Saving `reports/sarima_summary.txt`, `reports/sarima_forecast.csv`, and `models/sarima_212_1117_store1_grocery12.pkl`.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# --- 1. Universal Path Resolution ---
if os.path.exists('/content/drive/MyDrive/Sales-Forecasting_Project'):
    BASE_DIR = Path('/content/drive/MyDrive/Sales-Forecasting_Project/sales-forecasting')
else:
    cwd = Path.cwd().resolve()
    BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
REPORTS_DIR = BASE_DIR / 'reports'

for d in [MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Loading cleaned dataset from {DATA_DIR / 'clean_merged.csv'}...")
df = pd.read_csv(DATA_DIR / 'clean_merged.csv', parse_dates=['date'])
print(f"Loaded {df.shape[0]:,} rows.")


### 2. National Daily Series Preparation & Chronological Split
Aggregate sales to national daily totals, enforce frequency, and split 80% train / 20% test.


In [ ]:
daily_sales = df.groupby('date')['sales'].sum().reset_index().set_index('date').asfreq('D')
daily_sales['sales'] = daily_sales['sales'].interpolate(method='linear')

split_idx = int(len(daily_sales) * 0.8)
train = daily_sales.iloc[:split_idx]['sales']
test = daily_sales.iloc[split_idx:]['sales']

print(f"Train size: {len(train)} days | Test size: {len(test)} days")


### 3. Fitting SARIMA(2,1,2)(1,1,1)[7]
We fit seasonal order $(P=1, D=1, Q=1, s=7)$ combined with baseline order $(p=2, d=1, q=2)$.


In [ ]:
p, d, q = 2, 1, 2
P, D, Q, s = 1, 1, 1, 7

print(f"Fitting SARIMA({p},{d},{q})({P},{D},{Q})[{s}] on national sales...")
sarima_model = SARIMAX(
    train,
    order=(p, d, q),
    seasonal_order=(P, D, Q, s),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False)

print(sarima_fit.summary())


### 4. Forecasting & Diagnostic Visualization
Generate predictions over the test window along with 95% confidence intervals.


In [ ]:
forecast_obj = sarima_fit.get_forecast(steps=len(test))
predicted_mean = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int(alpha=0.05)

plt.figure(figsize=(15, 6))
plt.plot(train.index[-90:], train.iloc[-90:], label='Train (Recent 90 Days)', color='black', alpha=0.6)
plt.plot(test.index, test, label='Actual Test Sales', color='#1f77b4', linewidth=2)
plt.plot(test.index, predicted_mean, label=f'SARIMA({p},{d},{q})({P},{D},{Q})[7] Forecast', color='#2ca02c', linewidth=2, linestyle='--')
plt.fill_between(test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='#2ca02c', alpha=0.15, label='95% Confidence Interval')

plt.title('SARIMA National Daily Sales Forecast vs Actuals', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('National Sales')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 5. Robust Retail Error Metrics: WMAPE & SMAPE
Standard MAPE suffers from severe inflation or division-by-zero errors when sales approach zero.
In retail supply chain management:
- **WMAPE (Weighted MAPE):** $\frac{\sum |y - \hat{y}|}{\sum y} \times 100\%$
- **SMAPE (Symmetric MAPE):** $\frac{1}{n} \sum \frac{2|y - \hat{y}|}{|y| + |\hat{y}| + \epsilon} \times 100\%$


In [ ]:
mae = mean_absolute_error(test, predicted_mean)
rmse = np.sqrt(mean_squared_error(test, predicted_mean))
wmape = (np.sum(np.abs(test - predicted_mean)) / np.sum(test)) * 100
smape = np.mean(2.0 * np.abs(test - predicted_mean) / (np.abs(test) + np.abs(predicted_mean) + 1e-8)) * 100

print("=== SARIMA Evaluation Metrics ===")
print(f"MAE:   {mae:,.2f}")
print(f"RMSE:  {rmse:,.2f}")
print(f"WMAPE: {wmape:.2f}%")
print(f"SMAPE: {smape:.2f}%")


### 6. 5-Fold Walk-Forward Cross-Validation
To prevent optimistic leakage bias, we evaluate a walk-forward expanding window cross-validation across 5 folds with a 30-day forecast horizon.


In [ ]:
def walk_forward_cv(series, n_splits=5, horizon=30):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_metrics = []
    
    for fold, (train_idx, test_idx) in enumerate(tscv.split(series), 1):
        test_idx = test_idx[:horizon]
        y_train = series.iloc[train_idx]
        y_test = series.iloc[test_idx]
        
        # Fit seasonal model on fold train
        model = SARIMAX(y_train, order=(4, 1, 0), seasonal_order=(2, 1, 0, 7),
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        preds = fit.forecast(steps=len(y_test))
        
        fold_mae = mean_absolute_error(y_test, preds)
        fold_rmse = np.sqrt(mean_squared_error(y_test, preds))
        fold_metrics.append({'Fold': fold, 'MAE': fold_mae, 'RMSE': fold_rmse})
        print(f"Fold {fold}: MAE = {fold_mae:,.2f} | RMSE = {fold_rmse:,.2f}")
        
    avg_df = pd.DataFrame(fold_metrics)
    print("\n--- Cross-Validation Averages ---")
    print(f"Mean MAE:  {avg_df['MAE'].mean():,.2f}")
    print(f"Mean RMSE: {avg_df['RMSE'].mean():,.2f}")
    return avg_df

print("Running 5-fold walk-forward validation (horizon=30 days)...")
cv_results = walk_forward_cv(daily_sales['sales'], n_splits=5, horizon=30)


### 7. Exporting Artifacts
Serialize `reports/sarima_summary.txt`, `reports/sarima_forecast.csv`, and the store-level benchmark pickle.


In [ ]:
# 1. Text Summary
summary_path = REPORTS_DIR / 'sarima_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(str(sarima_fit.summary()))
print(f"Saved: {summary_path}")

# 2. Forecast CSV
forecast_df = pd.DataFrame({
    'date': test.index,
    'actual': test.values,
    'predicted': predicted_mean.values,
    'lower_ci': conf_int.iloc[:, 0].values,
    'upper_ci': conf_int.iloc[:, 1].values
})
forecast_path = REPORTS_DIR / 'sarima_forecast.csv'
forecast_df.to_csv(forecast_path, index=False)
print(f"Saved: {forecast_path}")

# 3. Model save
model_save_path = MODELS_DIR / f'sarima_{p}{d}{q}_{P}{D}{Q}{s}_store1_grocery12.pkl'
sarima_fit.save(model_save_path)
print(f"Saved model binary: {model_save_path}")
print("\nSARIMA pipeline completed successfully! Proceed to 06_prophet_forecasting.ipynb.")
